In [1]:
#CRIAÇÃO DA TABELA GOLD ADVANCED FACT INVENTORY EXPOSURE
from datetime import datetime
from pathlib import Path
import duckdb

# Procura o banco automaticamente : o notebook continuará encontrando o banco automaticamente dentro do projeto, 
# mesmo que ele seja movido para outra pasta ou máquina, desde que a estrutura do projeto seja mantida.
base_dir = Path.cwd().parent

db_path = next(base_dir.rglob("supply_chain_analytics.duckdb"))

conn = duckdb.connect(str(db_path))

# ============================================================
# CRIAÇÃO DA TABELA FATO DE EXPOSIÇÃO DE ESTOQUE
# ============================================================
#
# Objetivo:
# Consolidar informações de vendas, estoque e movimentações
# em uma única tabela analítica com granularidade semanal.
#
# Esta tabela fornece uma visão integrada da exposição do
# estoque ao longo do tempo, permitindo analisar a relação
# entre demanda, disponibilidade de produtos e fluxo logístico.
#
# Transformações realizadas:
# - Agregação semanal das vendas
# - Agregação semanal dos níveis de estoque
# - Agregação semanal das movimentações de estoque
# - Consolidação das métricas em uma única tabela fato
# - Preservação de registros mesmo quando existirem dados em
#   apenas uma das fontes (FULL OUTER JOIN)
#
# Esta tabela será utilizada para análises de:
#
# - Cobertura de estoque
# - Giro de estoque
# - Exposição ao risco de ruptura
# - Balanceamento de inventário
# - Fluxo logístico semanal
# - Planejamento de reposição
#
# ============================================================

conn.execute("""
CREATE OR REPLACE TABLE gold__fact_inventory_exposure__lite AS

-- ============================================================
-- AGREGAÇÃO SEMANAL DAS VENDAS
-- ============================================================
WITH weekly_sales AS (

    SELECT

        -- Início da semana da venda
        DATE_TRUNC('week', SalesDate) AS WeekStartDate,

        -- Localidade da venda
        LocationID,

        -- Produto vendido
        ProductSKU,

        -- Quantidade total vendida na semana
        SUM(Quantity) AS WeeklySalesQty

    FROM gold__fact_sales__lite

    GROUP BY 1,2,3
),

-- ============================================================
-- AGREGAÇÃO SEMANAL DOS SNAPSHOTS DE ESTOQUE
-- ============================================================
weekly_inventory AS (

    SELECT

        -- Início da semana do snapshot
        DATE_TRUNC('week', SnapshotDate) AS WeekStartDate,

        -- Localidade do estoque
        LocationID,

        -- Produto em estoque
        ProductSKU,

        -- Maior quantidade registrada na semana
        -- Representando o estoque disponível ao final do período
        MAX(OnHandQuantity) AS WeekEndOnHandQty

    FROM gold__fact_inventory_snapshots__lite

    GROUP BY 1,2,3
),

-- ============================================================
-- AGREGAÇÃO SEMANAL DAS MOVIMENTAÇÕES DE ESTOQUE
-- ============================================================
weekly_movements AS (

    SELECT

        -- Início da semana da movimentação
        DATE_TRUNC('week', MovementDate) AS WeekStartDate,

        -- Utiliza o local de destino quando disponível.
        -- Caso contrário, utiliza o local de origem.
        COALESCE(ToLocationID, FromLocationID) AS LocationID,

        -- Produto movimentado
        ProductSKU,

        -- Quantidade total movimentada na semana
        SUM(Quantity) AS NetMovementQty

    FROM gold__fact_inventory_movements__lite

    GROUP BY 1,2,3
)

-- ============================================================
-- CONSOLIDAÇÃO DAS MÉTRICAS SEMANAIS
-- ============================================================
SELECT

    -- Semana de referência consolidada
    COALESCE(
        s.WeekStartDate,
        i.WeekStartDate,
        m.WeekStartDate
    ) AS WeekStartDate,

    -- Localidade consolidada
    COALESCE(
        s.LocationID,
        i.LocationID,
        m.LocationID
    ) AS LocationID,

    -- Produto consolidado
    COALESCE(
        s.ProductSKU,
        i.ProductSKU,
        m.ProductSKU
    ) AS ProductSKU,

    -- Quantidade vendida na semana
    s.WeeklySalesQty,

    -- Estoque disponível ao final da semana
    i.WeekEndOnHandQty,

    -- Quantidade movimentada na semana
    m.NetMovementQty

FROM weekly_sales s

-- Une vendas e estoque
FULL OUTER JOIN weekly_inventory i
    ON s.WeekStartDate = i.WeekStartDate
   AND s.LocationID = i.LocationID
   AND s.ProductSKU = i.ProductSKU

-- Une movimentações à base consolidada
FULL OUTER JOIN weekly_movements m
    ON COALESCE(s.WeekStartDate, i.WeekStartDate) = m.WeekStartDate
   AND COALESCE(s.LocationID, i.LocationID) = m.LocationID
   AND COALESCE(s.ProductSKU, i.ProductSKU) = m.ProductSKU;
""")

# Verifica se a tabela foi criada com sucesso
tables = conn.sql("SHOW TABLES").df()


if "gold__fact_inventory_exposure__lite" in tables["name"].values:
    print(
        f"[{datetime.now():%d/%m/%Y %H:%M:%S}] "
        f"✅ Tabela gold__fact_inventory_exposure__lite criada com sucesso!"
    )
else:
    print(
        f"[{datetime.now():%d/%m/%Y %H:%M:%S}] "
        f"❌ Tabela gold__fact_inventory_exposure__lite não foi criada!"
    )
conn.close()

[04/06/2026 00:28:56] ✅ Tabela gold__fact_inventory_exposure__lite criada com sucesso!
